In [1]:
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load data

In [2]:
stats_3g = "3G-data-compression-stats.csv"
stats_13g = "13G-data-compression-stats.csv"
stats_134g = "134G-data-compression-stats.csv"


In [3]:
# Read and combine data stats
df_3g = pd.read_csv(stats_3g)
df_13g = pd.read_csv(stats_13g)
df_134g = pd.read_csv(stats_134g)

df_3g['dataset'] = '3G'
df_13g['dataset'] = '13G'
df_134g['dataset'] = '134G'

df_all = pd.concat([df_3g, df_13g, df_134g], ignore_index=True)

del df_3g, df_13g, df_134g

# Add branch prefix
df_all['branch_name_prefix'] = df_all['branch_name'].apply(lambda x: x.split('.')[0])

# Simplify branch types
branch_types = df_all['branch_type'].values
branch_type_map = {}

for branch_type in branch_types:
    if branch_type.startswith('DataVector<xAOD::'):
        branch_type_map[branch_type] = 'DataVector<xAOD::*>'
    elif branch_type.startswith('int'):
        branch_type_map[branch_type] = 'int'
    elif branch_type.startswith('std::vector<int'):
        branch_type_map[branch_type] = 'std::vector<int>'
    elif branch_type.startswith('std::vector<std::vector<ElementLink'):
        branch_type_map[branch_type] = 'std::vector<std::vector<ElementLink<*>>'
    elif branch_type.startswith('std::vector<std::vector<int'):
        branch_type_map[branch_type] = 'std::vector<std::vector<int*>>'
    elif branch_type.startswith('std::vector<uint'):
        branch_type_map[branch_type] = 'std::vector<uint*>'
    elif branch_type.startswith('uint'):
        branch_type_map[branch_type] = 'uint*'
    elif branch_type.startswith('vector<ElementLink<'):
        branch_type_map[branch_type] = 'vector<ElementLink<*>>'
    elif branch_type.startswith('vector<'):
        branch_type_map[branch_type] = 'vector<*>'
    elif branch_type.startswith('xAOD::'):
        branch_type_map[branch_type] = 'xAOD::*'
    else:
        branch_type_map[branch_type] = branch_type


df_all['branch_type_higher'] = df_all['branch_type'].map(branch_type_map)

# Add column determining if branch is std::vector<float>
df_all['is_std_vector_float'] = df_all['branch_type'].str.startswith('std::vector<float')

df_all.columns

Index(['filename', 'branch_name', 'branch_type', 'compressor',
       'compressed_bytes', 'uncompressed_bytes', 'compression_ratio',
       'dataset', 'branch_name_prefix', 'branch_type_higher',
       'is_std_vector_float'],
      dtype='object')

# What is in a `TTree`?

- 50 unique branch types
- 947 unique branches...

- `std::vector<float>` branches account for 22-23% of the branches in a `TTree`.
- The makes `std::vector<float>` branches the plurality of branches in a `TTree`.




In [4]:
# Count number of times branch appears across all datasets but not across all files in all datasets
# So branch counts should be between 1 and 3
branch_counts = df_all.groupby(['branch_name'])['dataset'].nunique().reset_index()
branch_counts

,branch_name,dataset
0,AnalysisElectrons,3
1,AnalysisElectronsAux.,3
2,AnalysisElectronsAuxDyn.DFCommonElectronsECIDS,3
3,AnalysisElectronsAuxDyn.DFCommonElectronsECIDS...,3
4,AnalysisElectronsAuxDyn.DFCommonElectronsLHLoose,3
...,...,...
942,egammaClustersAuxDyn.constituentClusterLinks,3
943,egammaClustersAuxDyn.e_sampl,3
944,egammaClustersAuxDyn.eta_sampl,3
945,index_ref,3


In [5]:
# Normalized value counts of 'branch_type' by 'dataset'
branch_type_counts = df_all.groupby(['dataset'])['branch_type'].value_counts(normalize=True).unstack(level=0)
branch_type_counts *= 100

# Sort by average across datasets
branch_type_counts.reindex(branch_type_counts.mean(axis=1).sort_values(ascending=False).index, axis=0)

dataset,134G,13G,3G
branch_type,,,
std::vector<float>,23.358183,24.890307,23.666619
xAOD::AuxContainerBase,19.883805,22.257679,21.277201
DataVector<xAOD::TrigComposite_v1>,17.686699,19.864380,19.001565
std::vector<std::vector<ElementLink<DataVector<xAOD::IParticle>>>>,14.370272,6.302353,10.681269
std::vector<uint8_t>,6.591317,7.179896,6.826909
std::vector<int8_t>,2.306961,2.512964,2.389418
std::vector<std::vector<float>>,1.867540,2.034304,1.934291
uint32_t,1.867540,2.034304,1.934291
float,1.318263,1.435979,1.365382


# How is lossless compression doing?

In [6]:
df_all['compression_ratio'].describe()

count    207548.000000
mean          7.998184
std           6.493774
min           1.000000
25%           3.808565
50%           5.004431
75%           8.288572
max          86.179427
Name: compression_ratio, dtype: float64

In [7]:
df_all.sort_values(by='compression_ratio', ascending=False, inplace=True)
df_all

,filename,branch_name,branch_type,compressor,compressed_bytes,uncompressed_bytes,compression_ratio,dataset,branch_name_prefix,branch_type_higher,is_std_vector_float
14628,DAOD_PHYSLITE.37019878._000012.pool.root.1,PrimaryVerticesAuxDyn.trackParticleLinks,std::vector<std::vector<ElementLink<DataVector...,ZSTD(5),15658698,1349457622,86.179427,13G,PrimaryVerticesAuxDyn,std::vector<std::vector<ElementLink<*>>,False
15392,DAOD_PHYSLITE.37019878._000013.pool.root.1,PrimaryVerticesAuxDyn.trackParticleLinks,std::vector<std::vector<ElementLink<DataVector...,ZSTD(5),15227885,1311879936,86.149845,13G,PrimaryVerticesAuxDyn,std::vector<std::vector<ElementLink<*>>,False
16436,DAOD_PHYSLITE.37019878._000014.pool.root.1,PrimaryVerticesAuxDyn.trackParticleLinks,std::vector<std::vector<ElementLink<DataVector...,ZSTD(5),14284406,1230103389,86.115124,13G,PrimaryVerticesAuxDyn,std::vector<std::vector<ElementLink<*>>,False
17063,DAOD_PHYSLITE.37019878._000015.pool.root.1,PrimaryVerticesAuxDyn.trackParticleLinks,std::vector<std::vector<ElementLink<DataVector...,ZSTD(5),13619764,1168224786,85.774231,13G,PrimaryVerticesAuxDyn,std::vector<std::vector<ElementLink<*>>,False
17997,DAOD_PHYSLITE.37019878._000016.pool.root.1,PrimaryVerticesAuxDyn.trackParticleLinks,std::vector<std::vector<ElementLink<DataVector...,ZSTD(5),13807743,1182972200,85.674552,13G,PrimaryVerticesAuxDyn,std::vector<std::vector<ElementLink<*>>,False
...,...,...,...,...,...,...,...,...,...,...,...
186062,DAOD_PHYSLITE.37019981._000238.pool.root.1,EventInfoAuxDyn.timeStampNSOffset,uint32_t,ZSTD(5),756792,756792,1.000000,134G,EventInfoAuxDyn,uint*,False
89505,DAOD_PHYSLITE.37019981._000078.pool.root.1,EventInfoAuxDyn.timeStampNSOffset,uint32_t,ZSTD(5),393844,393844,1.000000,134G,EventInfoAuxDyn,uint*,False
154173,DAOD_PHYSLITE.37019981._000174.pool.root.1,EventInfoAuxDyn.timeStampNSOffset,uint32_t,ZSTD(5),331698,331698,1.000000,134G,EventInfoAuxDyn,uint*,False
78821,DAOD_PHYSLITE.37019981._000066.pool.root.1,EventInfoAuxDyn.timeStampNSOffset,uint32_t,ZSTD(5),400158,400158,1.000000,134G,EventInfoAuxDyn,uint*,False


In [8]:
import plotly.express as px
import numpy as np

# Drop IQR outliers
q1 = df_all['compression_ratio'].quantile(0.25)
q3 = df_all['compression_ratio'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

plot_df = df_all.copy()
plot_df = df_all[(df_all['compression_ratio'] >= lower_bound) & (df_all['compression_ratio'] <= upper_bound)]

# Average compression ratios by dataset
# This is to reduce number of points in the plot because it's really struggling
plot_df = plot_df.groupby(['branch_type', 'branch_name_prefix', 'branch_name']).mean(numeric_only = True).reset_index()
plot_df['branch_type_higher'] = plot_df['branch_type'].map(branch_type_map)
plot_df['is_std_vector_float'] = plot_df['branch_type'].str.startswith('std::vector<float>')

fig1 = px.histogram(
    plot_df,
    title = 'Compression Ratios by Branch Type',
    subtitle = 'Average compression ratio for each branch type, excluding outliers',
    x = 'compression_ratio',
    color = 'is_std_vector_float',
    marginal = 'rug'
)

fig1.update_layout(
    legend = dict(
        title = 'Branch Type'
    ),
    margin = dict(t=60),
    xaxis = dict(title='Compression Ratio'),
    yaxis = dict(title='Count'),
)

fig2 = px.treemap(
    plot_df,
    title = 'Compressed Bytes by Branch Type, Branch Variable',
    path = ['branch_type_higher', 'branch_type', 'branch_name_prefix', 'branch_name'],
    values = 'compressed_bytes',
    color = 'compression_ratio',
)

fig2.update_layout(
    margin = dict(t=60),
    coloraxis_colorbar=dict(
        title='Compression Ratio',
    ),
)

fig1.write_html('compression_ratios_by_branch_type.html', include_plotlyjs='cdn')
fig2.write_html('compressed_bytes_by_branch_type.html', include_plotlyjs='cdn')

# fig1.show()
# fig2.show()